# CSA SCST+CE161 — kiểm tra năm caption trước reranking
    Chạy trên notebook của bạn, GPU **T4x2**, Internet ON. Attach **csa-full-roi-karpathy-v134**.
    Checkpoint chưa lưu Dataset: notebook tải **best.bin từ đúng version161**; bật Secret API
    và sửa SECRET_NAME trong cell đầu nếu tên không phải `codex`. Không cần Dataset checkpoint mới.

    Giữ nguyên mô hình và beam5/max20. Lấy đủ25.000 ứng viên trên VALID5000; kiểm câu đầu
    khớp kết quả161, chấm từng beam và phép chọn lý tưởng theo CIDEr tham chiếu.
    **Oracle dùng đáp án nên chỉ là chẩn đoán dư địa, không phải kết quả reranker triển khai.**
    Báo cả CHAIR/recall, số câu quá ngắn, độ đa dạng. Giữ output ứng viên để dùng cho reranker sau.
    Không train/TEST/ITM, không tải ảnh hay chạy lại CLIP, không stage TRAIN.2h là giới hạn dừng,
    không phải dự báo thời gian hoặc quota. User tự chạy; author chỉ kiểm metadata/NumPy/static.
    Save Version → Save & Run All. Gửi ZIP review sau khi chạy.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
FULL_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-roi-karpathy-v134')
CHECKPOINT_ROOT='' # Optional local mount/file. Empty downloads exact SaveVersion161 from your notebook.
KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/161'
SECRET_NAME='codex' # Set to the actual ENABLED Kaggle Secret name, e.g. KAGGLE_API_TOKEN.
WALL_HOURS=2 # STOP cap only, not expected runtime/billing. Latest user quota16h.
OUTPUT=Path('/kaggle/working/csa_ce161_candidates_valid5000')
if not FULL_ROOT.is_dir():raise FileNotFoundError('Attach fullROI134 Dataset before GPU: '+str(FULL_ROOT))
if OUTPUT.exists():raise RuntimeError('Preserve prior output; fresh output/session required.')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<2*1024**3:raise RuntimeError('Need2GiB free.')
GPU=json.loads(subprocess.check_output([sys.executable,'-c',"import json,torch;print(json.dumps(dict(built=torch.backends.cuda.is_built(),available=torch.cuda.is_available(),count=torch.cuda.device_count())))"],text=True,timeout=60))
if not GPU['built'] or not GPU['available'] or GPU['count']!=2:raise RuntimeError('Select T4x2 and restart runtime.')
OUTPUT.mkdir();SOURCE=Path(tempfile.mkdtemp(prefix='csa-candidates-'));LOG=OUTPUT/'notebook.log'
START=time.monotonic();DEADLINE=START+WALL_HOURS*3600
COMMIT='b37d996f1950766dfa579df8c823699d03ae1b5d'
HASHES={'candidate_data.py': 'c9f8cf09d2ab72372eb1fffdc4b56bb097b12e8dd46aac68c60d60b65cf04257', 'run_candidates.py': '689e73346cf19a36dc74afc1db7c29a8b8fd8e4ca05a354e4943805f5e85c729', 'test_candidate_data.py': 'ca481f6e1602f89ff5a83cc80326310b2ce41a8caf66db6b3af3320a2472d269', 'make_candidate_notebook.py': '8303fbad56cf11c5ab0db2a2174a3433ea43af1f14039593818380f61a8cc82f', 'CANDIDATE_BASELINE161.json.gz': '9c11ad2df2bd74784667125c318b0520f9c15b5e89158f76fb487139590e75d4', 'CANDIDATE_DIAGNOSTIC_PROTOCOL.md': 'fb3f6bbe7056450118ff0d2ac2682a69c2039bf51ab4b5c50a93ef8cdc0936f3', 'SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'make_smoke_notebook.py': '9de13e286efb3ddd29a2eed331219a9e60dcb384fc2f28e062f64dd8dd1c6054'}
for n,h in HASHES.items():
    raw=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{n}',timeout=90).read()
    if hashlib.sha256(raw).hexdigest()!=h:raise ValueError('Source differs: '+n)
    (SOURCE/n).write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES,gpu=GPU),indent=2),encoding='utf-8')
os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json';CHAIR=SOURCE/'chair'


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Candidate diagnostic active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    run([sys.executable,'-u',SOURCE/'test_candidate_data.py'],timeout=120)
    # Metadata fixture only; actual model is loaded exclusively by worker subprocesses.
    sys.path.insert(0,str(SOURCE))
    from candidate_data import CHECKPOINT_SHA
    from real_bank_data import sha
    if CHECKPOINT_ROOT:
        root=Path(CHECKPOINT_ROOT)
        found=[root] if root.is_file() else list(root.rglob('best.bin'))+list(root.rglob('checkpoint.bin'))
    else:
        if KERNEL_VERSION!='ducanh2403/2-vi-cai-kia-qua-nhieu/161':raise ValueError('No version/latest fallback.')
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret(SECRET_NAME).strip()
        if not token:raise RuntimeError('Enabled API Secret is empty.')
        environment=os.environ.copy()
        for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(key,None)
        environment['KAGGLE_API_TOKEN']=token;environment['PYTHONUNBUFFERED']='1'
        download=OUTPUT/'checkpoint161';download.mkdir()
        print('Downloading ONLY best.bin from exact161; no Dataset creation.',flush=True)
        command=[sys.executable,'-m','kaggle','kernels','output',KERNEL_VERSION,'-p',str(download),'--file-pattern',r'(^|/)best\.bin$']
        subprocess.run(command,env=environment,check=True,timeout=900)
        environment.pop('KAGGLE_API_TOKEN',None);del token,environment
        found=list(download.rglob('best.bin'))
    matches=[p for p in found if sha(p)==CHECKPOINT_SHA]
    if not matches:raise ValueError('Missing exact SCST+CE END161 weights; preserve output, no substitute.')
    checkpoint=sorted(matches)[0]
    (OUTPUT/'checkpoint_source.json').write_text(json.dumps(dict(source=KERNEL_VERSION if not CHECKPOINT_ROOT else CHECKPOINT_ROOT,sha256=CHECKPOINT_SHA)),encoding='utf-8')
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2','nltk==3.9.2','future'],timeout=300)
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],timeout=300)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],timeout=300)
    run(['java','-version'],timeout=30)
    run(['git','init',CHAIR],timeout=60)
    run(['git','-C',CHAIR,'remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],timeout=60)
    run(['git','-C',CHAIR,'fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],timeout=300)
    run(['git','-C',CHAIR,'checkout','--detach','FETCH_HEAD'],timeout=60)
    cfg=dict(full_root=str(FULL_ROOT),checkpoint=str(checkpoint),output=str(OUTPUT),stage=str(SOURCE/'stage'),chair=str(CHAIR))
    CONFIG.write_text(json.dumps(cfg),encoding='utf-8')
    run([sys.executable,'-u',SOURCE/'run_candidates.py','prepare','--config',CONFIG],timeout=min(900,DEADLINE-time.monotonic()))
    run([sys.executable,'-u',SOURCE/'run_candidates.py','metric_preflight','--config',CONFIG],timeout=min(600,DEADLINE-time.monotonic()))
    cfg['worker_seconds']=DEADLINE-time.monotonic()-1200
    if cfg['worker_seconds']<600:raise TimeoutError('Insufficient decode/metrics reserve; preserve output.')
    CONFIG.write_text(json.dumps(cfg),encoding='utf-8')
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    run([*launch,SOURCE/'run_candidates.py','worker','--config',CONFIG],timeout=cfg['worker_seconds']+60)
    run([sys.executable,'-u',SOURCE/'run_candidates.py','score','--config',CONFIG],timeout=min(1200,DEADLINE-time.monotonic()))
    result=json.loads((OUTPUT/'result.json').read_text(encoding='utf-8'))
    complete=result['status']=='CSA_CANDIDATE_DIAGNOSTIC_COMPLETE'
    if not complete:raise RuntimeError('Incomplete diagnostic')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,training=False,test_used=False,actual_reranking=False)),encoding='utf-8')
    archive=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',archive,flush=True)
    print('KEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('COMPLETE — oracle is diagnostic only; no trained/reranked result.' if complete else 'INCOMPLETE — preserve logs; no automatic retry.',flush=True)
